In [1]:
!pip install openai


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Я решил сначала установить доккер контейнер с LLM (максимального размера, который может тянуть моё железо), и и проверить соединение с ним.

In [2]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama")

response = client.chat.completions.create(
    model="qwen3:4b",
    messages=[{"role": "user", "content": "Привет"}])

print(response.choices[0].message.content)

Привет! 😊 Как дела?


# Банковский консультант: RAG - система

RAG-система создана для ответа по документам и условиям по брокерскому обслуживанию.
Документы скачаны с с сайта https://www.sberbank.ru/ru/person/investments/doc из раздела "Документы и условия по брокерскому обслуживанию" (большая часть,документов. Не скачивал zip приложения, doc файлы и pfd документы, которые представляют из себя таблицы. Фокус был на текстовые документы.)

### Этап №1. Подготовка данных и создание базы знаний

1. Устанавливаю библиотеки 

In [3]:
!pip install -qU langchain langchain-community langchain-text-splitters tiktoken nltk

import nltk
nltk.download('punkt_tab') # Обновленный пакет для предложений
nltk.download('punkt')

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter, NLTKTextSplitter



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
d:\SBER_RAG\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2. Загружаю документы из папки 'documents_SBER'

In [4]:
loader = DirectoryLoader('documents_SBER', glob = '*.txt', loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = loader.load()
print(f'Успешно загружено документов {len(documents)} \n')

Успешно загружено документов 8 



3. Применяем разные стратегии чанкинга (согласно в заданию)

In [5]:
# Стратегия А: По размеру (жестко рубит текст по символам и переносам строк)
char_splitter = CharacterTextSplitter(chunk_size=600, chunk_overlap=150, separator="\n")
char_chunks = char_splitter.split_documents(documents)

# Стратегия Б: Рекурсивный чанкинг (старается не рвать абзацы)
rec_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=150)
rec_chunks = rec_splitter.split_documents(documents)

# Стратегия В: По предложениям (использует лингвистические правила NLTK)
nltk_splitter = NLTKTextSplitter(chunk_size=600, chunk_overlap=150)
nltk_chunks = nltk_splitter.split_documents(documents)

Created a chunk of size 648, which is longer than the specified 600
Created a chunk of size 731, which is longer than the specified 600
Created a chunk of size 749, which is longer than the specified 600
Created a chunk of size 934, which is longer than the specified 600
Created a chunk of size 625, which is longer than the specified 600
Created a chunk of size 635, which is longer than the specified 600
Created a chunk of size 855, which is longer than the specified 600
Created a chunk of size 756, which is longer than the specified 600
Created a chunk of size 657, which is longer than the specified 600
Created a chunk of size 748, which is longer than the specified 600
Created a chunk of size 892, which is longer than the specified 600
Created a chunk of size 813, which is longer than the specified 600
Created a chunk of size 1085, which is longer than the specified 600
Created a chunk of size 617, which is longer than the specified 600
Created a chunk of size 601, which is longer th

4. Сравниваю результаты на примере первого куска (чанка)

In [6]:
print("--- СРАВНЕНИЕ СТРАТЕГИЙ ЧАНКИНГА ---\n")
if len(char_chunks) > 0:
    print(f"1. По размеру (Всего чанков: {len(char_chunks)}): \n{char_chunks[0].page_content}\n")
if len(rec_chunks) > 0:
    print(f"2. Рекурсивный (Всего чанков: {len(rec_chunks)}): \n{rec_chunks[0].page_content}\n")
if len(nltk_chunks) > 0:
    print(f"3. По предложениям (Всего чанков: {len(nltk_chunks)}): \n{nltk_chunks[0].page_content}\n")

--- СРАВНЕНИЕ СТРАТЕГИЙ ЧАНКИНГА ---

1. По размеру (Всего чанков: 2008): 
1
Правила применения рекомендательных технологий ПАО Сбербанк
Рекомендательные технологии – информационные технологии предоставления информации на основе сбора, систематизации и анализа сведений, относящихся к предпочтениям пользователей сети «Интернет», находящихся на территории Российской Федерации. Рекомендательные технологии позволяют рекомендовать продукты, товары и услуги, контент и прочие предложения, представляющие для пользователя потенциальный интерес.
Этапы процесса работы рекомендательной технологии:

2. Рекурсивный (Всего чанков: 2344): 
1
Правила применения рекомендательных технологий ПАО Сбербанк
Рекомендательные технологии – информационные технологии предоставления информации на основе сбора, систематизации и анализа сведений, относящихся к предпочтениям пользователей сети «Интернет», находящихся на территории Российской Федерации. Рекомендательные технологии позволяют рекомендовать продукты, тов

### Вывод по результатам сравнения стратегий чанкинга
При тестировании трёх стратегий на банковских документах выяснилось, что методы дают принципиально разные результаты при работе со структурированным текстом:

- Чанкинг по размеру (CharacterTextSplitter, всего чанков: 1957) — игнорирует визуальное форматирование, склеивая заголовки, списки и прочую информацию в сплошной нечитаемый монолит.

- Разделение по предложениям (Sentence Splitter, всего чанков: 1904) — критически ломает структуру банковских документов: алгоритм отрывает нумерацию списков (например, цифры 1. и 2.) от названий самих продуктов, что может привести к путанице при извлечении фактов.

- Рекурсивный чанкинг (RecursiveCharacterTextSplitter, всего чанков: 2279) — показал наилучший результат. Он бережно сохраняет логические абзацы, отступы и структуру маркированных списков, не разрывая смысловые блоки на полуслове.


#### Итог:
Для построения векторной базы знаний выбран RecursiveCharacterTextSplitter, так как он обеспечивает генеративную модель (LLM) максимально чистым, логичным и структурированным контекстом, что критически важно для точности RAG-системы.

### Этап №2. Эмбеддинги и Векторная база

In [7]:
# 1. Устанавливаем необходимые библиотеки
!pip install -qU sentence-transformers langchain-huggingface qdrant-client


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
import os
from tqdm import tqdm

In [9]:
from transformers import AutoModel
model = AutoModel.from_pretrained("intfloat/multilingual-e5-large")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6004.92it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# 2. Инициализируем модель эмбеддингов (intfloat/multilingual-e5-large)
model_name = "intfloat/multilingual-e5-large"
print(f"Загружаем модель {model_name}...")
embeddings = HuggingFaceEmbeddings(model_name=model_name)

Загружаем модель intfloat/multilingual-e5-large...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5973.88it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


#### 3. Определяем размерность вектора (для e5-large это 1024)
Такой большой размер выбран для улучшения качества поиска

In [11]:
embedding_dim = 1024

#### 4. Подключение к базе данных

In [12]:
# 4. Подключаемся к Qdrant (локальное хранение в папке ./qdrant_storage)
qdrant_client = QdrantClient(path="./qdrant_storage")  # сохраняет данные на диск

collection_name = "bank_documents"

# Если коллекция уже существует – удаляем, чтобы создать заново
if qdrant_client.collection_exists(collection_name):
    qdrant_client.delete_collection(collection_name)
    print(f"Старая коллекция '{collection_name}' удалена.")

# Создаём новую коллекцию с нужной размерностью вектора
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE)
)
print(f"Коллекция '{collection_name}' создана (размерность = {embedding_dim}).")

Старая коллекция 'bank_documents' удалена.
Коллекция 'bank_documents' создана (размерность = 1024).


#### 5. Подготовка данных: для каждого чанка вычисляем эмбеддинг и формируем payload

In [13]:
print("Вычисляем эмбеддинги и готовим точки для загрузки...")

# Сначала соберём чанки, сгруппированные по имени документа, чтобы нумеровать их внутри документа
chunks_by_doc = {}
for chunk in rec_chunks:
    # Извлекаем полный путь к файлу из метаданных
    source_path = chunk.metadata.get("source", "unknown.txt")
    doc_name = os.path.splitext(os.path.basename(source_path))[0]  # имя файла без расширения
    chunks_by_doc.setdefault(doc_name, []).append(chunk)

# Теперь пройдём по каждому документу и нумеруем его чанки
points = []
point_id = 0

for doc_name, chunks in chunks_by_doc.items():
    for idx, chunk in enumerate(chunks, start=1):
        # Получаем вектор эмбеддинга для текста чанка
        embedding_vector = embeddings.embed_query(chunk.page_content)
        
        # Формируем payload
        payload = {
            "document_name": doc_name,
            "chunk_index": idx,
            "text": chunk.page_content,          # сохраняем и сам текст (удобно для последующего вывода)
            "source": chunk.metadata.get("source", ""),
        }
        
        points.append(PointStruct(id=point_id, vector=embedding_vector, payload=payload))
        point_id += 1

Вычисляем эмбеддинги и готовим точки для загрузки...


#### 6. Загружаем точки в Qdrant (по батчам, но для простоты – все сразу)

In [14]:



print(f"Загружаем {len(points)} чанков в Qdrant...")
batch_size = 100
for i in tqdm(range(0, len(points), batch_size)):
    batch = points[i:i+batch_size]
    qdrant_client.upsert(collection_name=collection_name, points=batch)

print("ГОТОВО! Векторная база Qdrant успешно создана и сохранена.")
print(f"Всего документов (чанков) в коллекции: {qdrant_client.count(collection_name=collection_name).count}")

Загружаем 2344 чанков в Qdrant...


100%|██████████| 24/24 [00:15<00:00,  1.51it/s]

ГОТОВО! Векторная база Qdrant успешно создана и сохранена.
Всего документов (чанков) в коллекции: 2344


### Этап 3: Система ретрива (извлечения) и оценка качества поиска.

Ниже приведена реализация гибридного поиска, объединяющего косинусную близость (семантический поиск через Qdrant) и BM25 (ключевой поиск).
Для каждого документа вычисляется final_score по формуле:

'final_score(d) = 1/(k + rank_vector(d)) + 1/(k + rank_bm25(d))'

    где k=1 (можно менять),
    rank_vector и rank_bm25 – позиции документа (1‑based) в выдаче семантического поиска и BM25 соответственно. 
    Если документ отсутствует в одной из выдач, соответствующий член суммы равен 0.

По результатам отбираются 2 чанка с наибольшим final_score – они и будут поданы в LLM в качестве контекста.

3.1 Установка и импорт необходимых библиотек

In [15]:
!pip install -qU rank_bm25 nltk pymorphy3

import numpy as np
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
import pymorphy3
import pickle
import re
from nltk.corpus import stopwords
nltk.download('punkt_tab')
nltk.download('punkt')
nltk.download('stopwords')



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

3.2 Построение BM25‑индекса
Используем уже созданные чанки (rec_chunks). Для каждого чанка сохраняем:

- токенизированный текст (для BM25)

- оригинальный текст и метаданные (имя документа, индекс чанка, source)

- сопоставление (document_name, chunk_index) → point_id (point_id взят из загрузки в Qdrant)

In [16]:
# Загружаем стоп-слова (русские)
russian_stopwords = set(stopwords.words('russian'))

# Инициализируем лемматизатор pymorphy3
morph = pymorphy3.MorphAnalyzer()

# Функция предобработки текста
def preprocess_text(text: str) -> list:
    """
    Очищает текст, приводит к нижнему регистру, убирает пунктуацию,
    удаляет стоп-слова, выполняет лемматизацию.
    Возвращает список очищенных токенов (лемм).
    """
    # Приводим к нижнему регистру
    text = text.lower()
    # Удаляем пунктуацию (оставляем только буквы, цифры и пробелы)
    text = re.sub(r'[^\w\s]', ' ', text)
    # Разбиваем на слова
    words = text.split()
    # Обрабатываем каждое слово: лемматизация + фильтр стоп-слов
    clean_tokens = []
    for word in words:
        if word.isdigit() or len(word) <= 1:
            continue  # игнорируем числа и слишком короткие токены
        if word in russian_stopwords:
            continue
        # Лемматизация
        lemma = morph.parse(word)[0].normal_form
        clean_tokens.append(lemma)
    return clean_tokens

# Собираем информацию о чанках, загруженных в Qdrant
key_to_point_id = {}
documents_for_bm25 = []   # список объектов с текстом, метаданными и point_id
tokenized_corpus = []     # список списков очищенных токенов

print("Предобработка чанков для BM25...")

for point in points:
    payload = point.payload
    doc_name = payload["document_name"]
    chunk_idx = payload["chunk_index"]
    text = payload["text"]
    
    # Ключ для идентификации чанка
    key = (doc_name, chunk_idx)
    key_to_point_id[key] = point.id
    
    # Очистка и токенизация
    clean_tokens = preprocess_text(text)
    tokenized_corpus.append(clean_tokens)
    
    # Сохраняем объект, аналогичный Document, для удобного доступа
    documents_for_bm25.append({
        "page_content": text,
        "metadata": {
            "document_name": doc_name,
            "chunk_index": chunk_idx,
            "source": payload["source"],
            "point_id": point.id
        }
    })

# Создаём BM25-индекс
print("Строим BM25 индекс...")
bm25 = BM25Okapi(tokenized_corpus)

# Сохраняем BM25 индекс и вспомогательные данные в файл .pkl
bm25_data = {
    "bm25": bm25,
    "documents_for_bm25": documents_for_bm25,
    "tokenized_corpus": tokenized_corpus,
    "key_to_point_id": key_to_point_id
}

with open("bm25_index.pkl", "wb") as f:
    pickle.dump(bm25_data, f)

print(f"BM25 индекс создан на {len(documents_for_bm25)} чанках и сохранён в 'bm25_index.pkl'.")

Предобработка чанков для BM25...
Строим BM25 индекс...
BM25 индекс создан на 2344 чанках и сохранён в 'bm25_index.pkl'.


#### 3.3 Загрузка BM25-индекса и вспомогательных данных из .pkl

In [17]:
with open("bm25_index.pkl", "rb") as f:
    bm25_data = pickle.load(f)

bm25 = bm25_data["bm25"]                      # готовый BM25Okapi объект
documents_for_bm25 = bm25_data["documents_for_bm25"]   # список метаданных чанков
tokenized_corpus = bm25_data["tokenized_corpus"]       # токенизированный корпус (необязательно)
key_to_point_id = bm25_data["key_to_point_id"]         # отображение (документ, индекс) -> point_id

print(f"BM25 индекс загружен. Всего чанков: {len(documents_for_bm25)}")

BM25 индекс загружен. Всего чанков: 2344


3.3 Функция гибридного поиска
Выполняет:

- семантический поиск через Qdrant (top_k = 3)

- BM25 поиск (top_k = 3)

- объединяет результаты, вычисляет final_score

- возвращает два лучших чанка (текст + метаданные)

In [35]:
def hybrid_search(query, k_vector=10, k_bm25=10, k_final=15, k_param=60):
    """
    Гибридный поиск: семантический (Qdrant) + BM25.
    Возвращает список из k_final словарей с ключами:
        - text: исходный текст чанка
        - metadata: метаданные (документ, индекс, источник, point_id)
        - final_score: вычисленный score
    """
    # ----- Семантический поиск (Qdrant) -----
    query_vector = embeddings.embed_query(query)
    
    # Автоматически выбираем метод поиска в зависимости от версии клиента
    response = qdrant_client.query_points(
    collection_name=collection_name,
    query=query_vector,  # Вектор запроса
    limit=k_vector
    )
    vector_results = response.points

    
    rank_vector = {}
    for rank, point in enumerate(vector_results, start=1):
        rank_vector[point.id] = rank

    # ----- Поиск BM25 -----
    # Очищаем запрос так же, как очищали чанки
    query_tokens = preprocess_text(query)
    if not query_tokens:          # если после очистки не осталось токенов – BM25 выдаст пустой результат
        bm25_scores = [0.0] * len(documents_for_bm25)
    else:
        bm25_scores = bm25.get_scores(query_tokens)

    # Индексы топ-k_bm25 чанков (индексы в списке documents_for_bm25)
    top_bm25_indices = np.argsort(bm25_scores)[::-1][:k_bm25]
    rank_bm25 = {}
    for rank, idx in enumerate(top_bm25_indices, start=1):
        point_id = documents_for_bm25[idx]["metadata"]["point_id"]
        rank_bm25[point_id] = rank

    # ----- Объединение и вычисление final_score -----
    all_point_ids = set(rank_vector.keys()) | set(rank_bm25.keys())
    scores = {}
    for pid in all_point_ids:
        rv = rank_vector.get(pid, None)
        rb = rank_bm25.get(pid, None)
        score = 0.0
        if rv is not None:
            score += 1.0 / (k_param + rv)
        if rb is not None:
            score += 1.0 / (k_param + rb)
        scores[pid] = score

    # Сортируем point_id по убыванию score
    sorted_pids = sorted(scores.keys(), key=lambda pid: scores[pid], reverse=True)[:k_final]

    # Формируем результат, извлекая текст и метаданные из documents_for_bm25
    result_chunks = []
    for pid in sorted_pids:
        # Ищем чанк по point_id
        for doc in documents_for_bm25:
            if doc["metadata"]["point_id"] == pid:
                result_chunks.append({
                    "text": doc["page_content"],
                    "metadata": doc["metadata"],
                    "final_score": scores[pid]
                })
                break
    return result_chunks

##### 3.4 Пример использования

In [48]:
test_query = "Какие правила применения рекомендательных технологий?"
context_chunks = hybrid_search(test_query, k_final=2)

print(f"Запрос: {test_query}\n")
for i, chunk in enumerate(context_chunks, 1):
    print(f"--- Чанк {i} (final_score = {chunk['final_score']:.4f}) ---")
    print(f"Документ: {chunk['metadata']['document_name']}, чанк #{chunk['metadata']['chunk_index']}")
    print(f"Текст (первые 500 символов):\n{chunk['text'][:500]}...\n")

Запрос: Какие правила применения рекомендательных технологий?

--- Чанк 1 (final_score = 0.0328) ---
Документ: 04102023_recommendation_system_rules, чанк #1
Текст (первые 500 символов):
1
Правила применения рекомендательных технологий ПАО Сбербанк
Рекомендательные технологии – информационные технологии предоставления информации на основе сбора, систематизации и анализа сведений, относящихся к предпочтениям пользователей сети «Интернет», находящихся на территории Российской Федерации. Рекомендательные технологии позволяют рекомендовать продукты, товары и услуги, контент и прочие предложения, представляющие для пользователя потенциальный интерес.
Этапы процесса работы рекомендате...

--- Чанк 2 (final_score = 0.0323) ---
Документ: 04102023_recommendation_system_rules, чанк #2
Текст (первые 500 символов):
Этапы процесса работы рекомендательной технологии:
1. Сбор сведений о пользователе осуществляется из различных автоматизированных систем и информационных ресурсов Банка. Рекомендательные

##### 3.5 Оценка качества поиска
Для оценки используем метрики:
- Hit Rate (попадание нужного документа в топ‑k)
- MRR (средняя обратная позиция)

Необходимо создать тестовый датасет, где каждый элемент – это (вопрос, ожидаемое_имя_документа).

Можно автоматически сгенерировал вопросы, но для объективной оценки лучше составить вручную.

In [20]:
!pip install -qU pandas openpyxl


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import pandas as pd
import re

# Загружаем Excel-файл (предполагается, что он лежит в текущей папке)
df = pd.read_excel('Test_dataset.xlsx', engine='openpyxl')

# Извлекаем колонки: вопрос и ожидаемый абзац-ответ
test_queries = df['Запрос пользователя'].tolist()
expected_answers = df['Абзац документа (ответ)'].tolist()

# Очищаем от возможных NaN и приводим к строке
test_dataset = [
    (str(q).strip(), str(a).strip()) 
    for q, a in zip(test_queries, expected_answers) 
    if pd.notna(q) and pd.notna(a)
]

print(f"Загружено {len(test_dataset)} тестовых пар.")

Загружено 79 тестовых пар.


#### 3.5.1 Оценка по перекрытию

Функция для вычисления максимального перекрытия

In [36]:
def longest_common_substring_length(s1: str, s2: str) -> int:
    """Возвращает длину наибольшей общей подстроки между s1 и s2."""
    if not s1 or not s2:
        return 0
    # Оптимизация для коротких строк – можно использовать суффиксные алгоритмы,
    # но для абзацев до 1000 символов квадратичный алгоритм подходит.
    len1, len2 = len(s1), len(s2)
    # Создаём матрицу (можно оптимизировать, храня только две строки)
    dp = [[0] * (len2 + 1) for _ in range(len1 + 1)]
    max_len = 0
    for i in range(1, len1 + 1):
        for j in range(1, len2 + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
                if dp[i][j] > max_len:
                    max_len = dp[i][j]
            else:
                dp[i][j] = 0
    return max_len

def text_overlap_ratio(expected: str, chunk_text: str, threshold=0.7) -> bool:
    """
    Возвращает True, если доля перекрытия между expected и chunk_text >= threshold.
    Перекрытие = (длина наибольшей общей подстроки) / max(длина expected, длина chunk_text)
    """
    if not expected or not chunk_text:
        return False
    lcs_len = longest_common_substring_length(expected, chunk_text)
    max_len = max(len(expected), len(chunk_text))
    overlap = lcs_len / max_len
    return overlap >= threshold

Улучшенная функция оценки с использованием перекрытия

In [37]:
def evaluate_hybrid_retriever(dataset, k=3, overlap_threshold=0.6):
    """
    Оценивает гибридный поиск.
    dataset: список кортежей (query, expected_answer_text) – оригинальные тексты из Excel
    k: количество первых результатов (чанков)
    overlap_threshold: минимальная доля перекрытия для признания чанка релевантным
    Возвращает hit_rate и mrr.
    """
    hits = 0
    reciprocal_ranks = []
    
    for query, expected in dataset:
        results = hybrid_search(query, k_final=k)
        
        found_positions = []
        for pos, chunk in enumerate(results, start=1):
            chunk_text = chunk['text']
            # Сравниваем оригинальные тексты (без нормализации)
            if text_overlap_ratio(expected, chunk_text, overlap_threshold):
                found_positions.append(pos)
        
        if found_positions:
            hits += 1
            reciprocal_ranks.append(1.0 / found_positions[0])
        else:
            reciprocal_ranks.append(0.0)
    
    hit_rate = hits / len(dataset)
    mrr = np.mean(reciprocal_ranks)
    return hit_rate, mrr

Загружаю датасет

In [38]:
df = pd.read_excel('Test_dataset.xlsx', engine='openpyxl')
test_queries = df['Запрос пользователя'].tolist()
expected_answers = df['Абзац документа (ответ)'].tolist()
test_dataset = [(str(q).strip(), str(a).strip()) for q, a in zip(test_queries, expected_answers) if pd.notna(q) and pd.notna(a)]
print(f"Загружено {len(test_dataset)} тестовых пар.")

Загружено 79 тестовых пар.


In [25]:
# Оценка с перекрытием по подстроке
hit_rate, mrr = evaluate_hybrid_retriever(test_dataset, k=10, overlap_threshold=0.6)
print(f"Hit Rate: {hit_rate:.2f}, MRR: {mrr:.3f}")

Hit Rate: 0.35, MRR: 0.264


#### 3.5.2 Альтернативный вариант – сравнение по триграммам (более устойчиво к разрывам)

Перед сравнениием по триграммам необходимо нормализовать текст эталонного ответа и текст из найденных чанков 

In [39]:
def normalize_text(text: str) -> str:
    """
    Приводит текст к нижнему регистру, удаляет пунктуацию,
    заменяет любую последовательность пробелов на один пробел.
    """
    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    # Удаляем всё, кроме букв, цифр и пробелов
    text = re.sub(r'[^\w\s]', ' ', text)
    # Сжимаем множественные пробелы
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [40]:
# Извлечение триграмм из нормализованного текста

def get_trigrams(text: str) -> set:
    """Возвращает множество триграмм (символьных n-грамм) из нормализованного текста."""
    if len(text) < 3:
        return set()
    return {text[i:i+3] for i in range(len(text)-2)}


# Сравнение через Jaccard по триграммам (с нормализацией)
def trigram_similarity(expected: str, chunk_text: str, threshold=0.6) -> bool:
    """
    Нормализует оба текста, вычисляет Jaccard similarity по триграммам
    и возвращает True, если значение >= threshold.
    """
    # Нормализация
    norm_expected = normalize_text(expected)
    norm_chunk = normalize_text(chunk_text)
    
    # Триграммы
    trigrams_exp = get_trigrams(norm_expected)
    trigrams_chunk = get_trigrams(norm_chunk)
    
    if not trigrams_exp or not trigrams_chunk:
        return False
    
    intersection = len(trigrams_exp & trigrams_chunk)
    union = len(trigrams_exp | trigrams_chunk)
    jaccard = intersection / union
    return jaccard >= threshold


In [41]:
def evaluate_hybrid_retriever_trigram(dataset, k=5, threshold=0.5):
    """
    Оценивает гибридный поиск.
    dataset: список кортежей (query, expected_answer_text)
    k: количество первых результатов (чанков)
    threshold: минимальный Jaccard для признания чанка релевантным
    Возвращает hit_rate и mrr.
    """
    hits = 0
    reciprocal_ranks = []
    
    for query, expected in dataset:
        results = hybrid_search(query, k_final=k)
        
        found_positions = []
        for pos, chunk in enumerate(results, start=1):
            chunk_text = chunk['text']
            if trigram_similarity(expected, chunk_text, threshold):
                found_positions.append(pos)
        
        if found_positions:
            hits += 1
            reciprocal_ranks.append(1.0 / found_positions[0])
        else:
            reciprocal_ranks.append(0.0)
    
    hit_rate = hits / len(dataset)
    mrr = np.mean(reciprocal_ranks)
    return hit_rate, mrr

In [42]:
# Загрузка тестового датасета из Excel
df = pd.read_excel('Test_dataset.xlsx', engine='openpyxl')
test_queries = df['Запрос пользователя'].tolist()
expected_answers = df['Абзац документа (ответ)'].tolist()
test_dataset = [(str(q).strip(), str(a).strip()) for q, a in zip(test_queries, expected_answers) if pd.notna(q) and pd.notna(a)]
print(f"Загружено {len(test_dataset)} тестовых пар.")

Загружено 79 тестовых пар.


In [43]:
k_eval = 5
jaccard_threshold = 0.5   # можно менять (0.5 – мягче, 0.7 – строже)

hit_rate, mrr = evaluate_hybrid_retriever_trigram(test_dataset, k=k_eval, threshold=jaccard_threshold)

print("="*50)
print(f"Оценка качества гибридного поиска (top-{k_eval} чанков, нормализованные триграммы, порог={jaccard_threshold})")
print(f"Тестовых запросов: {len(test_dataset)}")
print(f"Hit Rate@{k_eval}: {hit_rate:.2f} ({hit_rate*100:.0f}%)")
print(f"MRR: {mrr:.3f}")
print("="*50)

Оценка качества гибридного поиска (top-5 чанков, нормализованные триграммы, порог=0.5)
Тестовых запросов: 79
Hit Rate@5: 0.52 (52%)
MRR: 0.445


Замеряю

In [44]:
import numpy as np

def evaluate_hybrid_retriever(dataset, k=3):
    """
    Оценивает гибридный поиск.
    dataset: список кортежей (query, expected_answer_text)
    k: количество первых результатов (чанков), учитываемых при оценке
    Возвращает hit_rate и mrr.
    """
    hits = 0
    reciprocal_ranks = []
    
    for query, expected in dataset:
        # Получаем топ-k чанков от гибридного поиска
        results = hybrid_search(query, k_final=k)
        
        # Нормализуем ожидаемый ответ для сравнения
        norm_expected = normalize_text(expected)
        
        found_positions = []  # позиции (1-based) чанков, содержащих ожидаемый текст
        for pos, chunk in enumerate(results, start=1):
            norm_chunk = normalize_text(chunk['text'])
            # Проверяем, содержится ли ожидаемый текст в чанке (или чанк в ожидаемом тексте)
            if norm_expected in norm_chunk or norm_chunk in norm_expected:
                found_positions.append(pos)
        
        # Если есть хотя бы одно совпадение – считаем hit
        if found_positions:
            hits += 1
            # Reciprocal Rank – позиция первого совпадения
            reciprocal_ranks.append(1.0 / found_positions[0])
        else:
            reciprocal_ranks.append(0.0)
    
    hit_rate = hits / len(dataset)
    mrr = np.mean(reciprocal_ranks)
    return hit_rate, mrr

In [45]:
k_eval = 10
hit_rate, mrr = evaluate_hybrid_retriever(test_dataset, k=k_eval)

print("="*50)
print(f"Оценка качества гибридного поиска (top-{k_eval} чанков)")
print(f"Тестовых запросов: {len(test_dataset)}")
print(f"Hit Rate@{k_eval}: {hit_rate:.2f} ({hit_rate*100:.0f}%)")
print(f"MRR: {mrr:.3f}")
print("="*50)

Оценка качества гибридного поиска (top-10 чанков)
Тестовых запросов: 79
Hit Rate@10: 0.59 (59%)
MRR: 0.423


Результаты метрик получились довольно низкими, вероятнее всего это связано с генерацией вопросов LLM-кой. Люди лучше бы задавали вопросы и метрики выросли бы.

## Этап 4. Интеграция с LLM

In [54]:
import requests
import json

# -------------------------------------------------------------------
# 1. Настройки Ollama
# -------------------------------------------------------------------
OLLAMA_URL = "http://localhost:11434"
MODEL_NAME = "qwen3:4b"

# -------------------------------------------------------------------
# 2. Глобальная память для диалога
# -------------------------------------------------------------------
chat_history = []  # каждый элемент: (вопрос пользователя, ответ ассистента)

# -------------------------------------------------------------------
# 3. Функция вызова Ollama (генерация ответа)
# -------------------------------------------------------------------
def generate_ollama(prompt: str, system: str = None, max_tokens: int = 512) -> str:
    """
    Отправляет запрос к Ollama API и возвращает текст ответа.
    """
    data = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3,
            "top_p": 0.9,
            "max_tokens": max_tokens
        }
    }
    if system:
        data["system"] = system

    response = requests.post(f"{OLLAMA_URL}/api/generate", json=data)
    response.raise_for_status()
    result = response.json()
    return result.get("response", "").strip()

# -------------------------------------------------------------------
# 4. Основная функция RAG-бота с памятью
# -------------------------------------------------------------------
def ask_bot_with_memory(user_query: str, top_k: int = 5) -> str:
    global chat_history

    # 4.1 Умный поиск с учётом предыдущего вопроса (для уточнений)
    search_query = user_query
    if chat_history:
        last_question = chat_history[-1][0]
        search_query = f"{last_question} {user_query}"

    # 4.2 Гибридный поиск (функция hybrid_search уже определена в Этапе 3)
    retrieved_chunks = hybrid_search(search_query, k_final=top_k)

    # Формируем контекст и собираем источники
    context_parts = []
    sources = set()
    for chunk in retrieved_chunks:
        context_parts.append(chunk['text'])
        doc_name = chunk['metadata'].get('document_name', '')
        if doc_name:
            sources.add(doc_name)

    context = "\n\n---\n\n".join(context_parts) if context_parts else ""

    # 4.3 Формируем историю диалога (последние 2 оборота)
    history_text = ""
    for q, a in chat_history[-2:]:
        history_text += f"Вопрос: {q}\nОтвет: {a}\n\n"

    # 4.4 Системная инструкция (защита от галлюцинаций)
    system_prompt = (
        "Ты — вежливый консультант банка. Ответ строй строго на основе информации из предоставленного контекста."
        "Если в контексте есть реливантный ответ на вопрос (даже частично), то структурируй эту информацию и дай пользователю с указанием источников данных. Если информация в контексте никак не помогает пользователю, то отвечай: "
        "\"К сожалению, в моей базе знаний нет информации по этому вопросу. Обратитесь на горячую линию.\" "
        "Не выдумывай факты."
    )

    # 4.5 Промпт пользователя (контекст + история + текущий вопрос)
    user_prompt = f"""Контекст:
{context}

{history_text}Вопрос пользователя: {user_query}

Ответ (только на основе контекста):"""

    # 4.6 Генерация ответа через Ollama
    print(f"КЛИЕНТ: {user_query}")
    print("Консультант думает...")
    answer = generate_ollama(user_prompt, system=system_prompt, max_tokens=512)

    # Если ответ пустой (на всякий случай)
    if not answer:
        answer = "Не удалось получить ответ от модели."

    # 4.7 Добавляем источники, если они есть
    if sources:
        answer += f"\n\nИсточники: {', '.join(sources)}"

    # 4.8 Сохраняем диалог в историю
    chat_history.append((user_query, answer))

    return answer


#### 4. Примеры использования (можно запустить для проверки)

##### Пример №1

In [55]:
# Вопрос 1: Задаём контекст
ans1 = ask_bot_with_memory("Что такое необеспеченные сделки и при каких условиях брокер их разрешает?")
print(f"\nБОТ:\n{ans1}\n")
print("-" * 50 + "\n")

# Вопрос 2: Уточняющий вопрос (проверяем память)
ans2 = ask_bot_with_memory("А какие риски с ними связаны? Я имею в виду, что может произойти, если цена пойдёт не в мою сторону?")
print(f"\nБОТ:\n{ans2}\n")
print("-" * 50 + "\n")


КЛИЕНТ: Что такое необеспеченные сделки и при каких условиях брокер их разрешает?
Консультант думает...

БОТ:
Необеспеченные сделки — это сделки, в результате которых возникает необеспеченная позиция, то есть имущество клиента, переданное брокеру, недостаточно для исполнения обязательств с учетом ранее заключенных сделок. 

Брокер разрешает такие сделки **при соблюдении максимального «плеча»** (соотношения обязательств клиента по заключенным в его интересах сделкам и имущества), как это регулируется нормативными актами. 

Источник данных: контекст, раздел "Цель настоящей Декларации — предоставить вам информацию об основных рисках, с которыми связаны необеспеченные сделки (то есть сделки, в результате которых возникает необеспеченная позиция – для исполнения обязательств по которым на момент заключения сделки имущества клиента, переданного брокеру, недостаточно с учетом иных ранее заключенных сделок). Данные сделки подходят не всем клиентам. Нормативные акты ограничивают риски клиентов 

In [49]:
chat_history = []

##### Пример №2

In [50]:
# Вопрос 1: Задаём контекст
ans1 = ask_bot_with_memory("Что такое сделка ОТС-РЕПО и как она заключается через вашего брокера?")
print(f"\nБОТ:\n{ans1}\n")
print("-" * 50 + "\n")

# Вопрос 2: Уточняющий вопрос (проверяем память)
ans2 = ask_bot_with_memory("Можно ли её закрыть досрочно? И что для этого нужно сделать?")
print(f"\nБОТ:\n{ans2}\n")
print("-" * 50 + "\n")

КЛИЕНТ: Что такое сделка ОТС-РЕПО и как она заключается через вашего брокера?
Консультант думает...

БОТ:
Сделка ОТС-РЕПО — это договор (сделка РЕПО), по которому Продавец обязуется в установленный срок передать Пакет Ценных бумаг Покупателю, а Покупатель обязуется принять Пакет Ценных бумаг и уплатить Сумму покупки, а также в установленный срок передать Пакет Ценных бумаг Продавцу и уплатить Сумму выкупа. 

Через брокера (Банк) Сделка ОТС-РЕПО заключается следующим образом: Банк совершает сделку от своего имени, но за счет и по поручению Инвестора на внебиржевом рынке, определяя Контрагента по Сделке ОТС-РЕПО самостоятельно, если Инвестор не указал в Заявке на совершение Сделки ОТС-РЕПО третье лицо, и заключает её на условиях соглашения между Банком и Контрагентом.

Источники: risk_declaration, brokerage_agreement

--------------------------------------------------

КЛИЕНТ: Можно ли её закрыть досрочно? И что для этого нужно сделать?
Консультант думает...

БОТ:
Да, сделку ОТС-РЕПО мож

In [56]:
chat_history = []

##### Пример №3

In [57]:
# Вопрос 1: Задаём контекст
ans1 = ask_bot_with_memory("Можно ли вывести деньги с брокерского счёта, не дожидаясь расчётов по проданным ценным бумагам?")
print(f"\nБОТ:\n{ans1}\n")
print("-" * 50 + "\n")

# Вопрос 2: Уточняющий вопрос (проверяем память)
ans2 = ask_bot_with_memory("Как это работает и какой процент я заплачу за такую операцию?")
print(f"\nБОТ:\n{ans2}\n")
print("-" * 50 + "\n")

КЛИЕНТ: Можно ли вывести деньги с брокерского счёта, не дожидаясь расчётов по проданным ценным бумагам?
Консультант думает...

БОТ:
Да, можно вывести деньги с брокерского счёта, не дожидаясь расчётов по проданным ценным бумагам, при условии использования сервиса «Вывод, не дожидаясь расчетов». 

**Условия и детали:**  
- Сервис доступен Инвесторам, которые совершили сделки по продаже ценных бумаг и/или валюты (п. 13.9.1).  
- Для активации сервиса Инвестор направляет распоряжение на вывод денежных средств, сумма которого не превышает Плановую позицию по денежным средствам и свыше Свободного остатка на Торговом брокерском счете (п. 13.9.3).  
- Банк транслирует в мобильном приложении максимально доступную сумму для вывода с использованием данного сервиса.  

**Источники данных:**  
- Пункт 13.9.1 (условия предоставления сервиса).  
- Пункт 13.9.3 (условия согласия на использование сервиса и трансляция доступной суммы).  

К сожалению, в моей базе знаний нет информации по этому вопросу. 

In [53]:
chat_history = []

##### Пример №4

In [ ]:
# Вопрос 1: Задаём контекст
ans1 = ask_bot_with_memory("Как брокер определяет, являюсь ли я налоговым резидентом РФ?")
print(f"\nБОТ:\n{ans1}\n")
print("-" * 50 + "\n")

# Вопрос 2: Уточняющий вопрос (проверяем память)
ans2 = ask_bot_with_memory("Если я изменю статус, могу ли я вернуть излишне удержанный налог?")
print(f"\nБОТ:\n{ans2}\n")
print("-" * 50 + "\n")

КЛИЕНТ: Как брокер определяет, являюсь ли я налоговым резидентом РФ?
Консультант думает...

БОТ:
К сожалению, в моей базе знаний нет информации по этому вопросу. Обратитесь на горячую линию.

Источники: brokerage_agreement_iia, brokerage_agreement

--------------------------------------------------

КЛИЕНТ: Если я изменю статус, могу ли я вернуть излишне удержанный налог?
Консультант думает...

БОТ:
Да, если вы измените налоговый статус (например, с налогового резидента РФ на налогового нерезидента РФ или предоставите сведения о статусе налогового резидента иностранного государства, у которого заключен СОИДН с РФ), Банк осуществит перерасчет налогооблагаемой базы и возврат излишне удержанного налога. Это регулируется **Условиями предоставления брокерских и иных услуг ПАО Сбербанк** и **статьей 207 Налогового кодекса Российской Федерации**.  

Для возврата необходимо направить заявление об изменении налогового статуса (Приложение 15 к Условиям) в Банк после рекомендуемой даты, а также п

### ЭТАП 4: ОПТИМИЗАЦИЯ ПРОИЗВОДИТЕЛЬНОСТИ (Кеширование)

In [59]:
import time

print("ЭТАП 4: ОПТИМИЗАЦИЯ ПРОИЗВОДИТЕЛЬНОСТИ (Кеширование)")

# Создаем простой словарь для кеша
response_cache = {}

def ask_with_cache(user_query):
    # Проверяем, есть ли уже готовый ответ в кеше
    if user_query in response_cache:
        print("[КЕШ] Найден готовый ответ! Отдаем моментально")
        return response_cache[user_query]

    # Если в кеше нет, запускаем нашу RAG-систему
    print("[ГЕНЕРАЦИЯ] Ответа в кеше нет. Запускаем нейросеть")

    answer = ask_bot_with_memory(user_query)

    # Сохраняем результат в кеш
    response_cache[user_query] = answer
    return answer

# ТЕСТИРУЕМ СКОРОСТЬ
test_query = "Можно ли вывести деньги с брокерского счёта, не дожидаясь расчётов по проданным ценным бумагам?"

print(f"\nВОПРОС: {test_query}\n")

ЭТАП 4: ОПТИМИЗАЦИЯ ПРОИЗВОДИТЕЛЬНОСТИ (Кеширование)

ВОПРОС: Можно ли вывести деньги с брокерского счёта, не дожидаясь расчётов по проданным ценным бумагам?



In [60]:
# Запуск 1: Без кеша (Первый раз задаем вопрос)
start_time = time.time()
ans1 = ask_with_cache(test_query)
end_time = time.time()
time_without_cache = end_time - start_time

print(f"\nОТВЕТ (Запуск 1):\n{ans1}")
print(f"Время выполнения БЕЗ кеша: {time_without_cache:.2f} сек.\n")
print("-" * 50)

[ГЕНЕРАЦИЯ] Ответа в кеше нет. Запускаем нейросеть
КЛИЕНТ: Можно ли вывести деньги с брокерского счёта, не дожидаясь расчётов по проданным ценным бумагам?
Консультант думает...

ОТВЕТ (Запуск 1):
Да, можно вывести деньги с брокерского счёта, не дожидаясь расчётов по проданным ценным бумагам. Это регулируется **разделом 13.9.1 Условий предоставления брокерских и иных услуг ПАО Сбербанк**, согласно которому банк предоставляет Инвесторам, которые совершили сделки по продаже ценных бумаг и/или валюты, возможность вывести денежные средства с Торгового брокерского счета без ожидания проведения расчётов по этим сделкам при согласии на использование сервиса «Вывод, не дожидаясь расчетов». 

**Источник данных**: раздел 13.9.1 контекста (brokerage_agreement).

Источники: brokerage_agreement
Время выполнения БЕЗ кеша: 117.22 сек.

--------------------------------------------------


In [61]:
# Запуск 2: С кешем (Задаем ТОТ ЖЕ вопрос во второй раз)
start_time_2 = time.time()
ans2 = ask_with_cache(test_query)
end_time_2 = time.time()
time_with_cache = end_time_2 - start_time_2

print(f"\nОТВЕТ (Запуск 2):\n{ans2}")
print(f"Время выполнения С кешем: {time_with_cache:.5f} сек.\n")

[КЕШ] Найден готовый ответ! Отдаем моментально

ОТВЕТ (Запуск 2):
Да, можно вывести деньги с брокерского счёта, не дожидаясь расчётов по проданным ценным бумагам. Это регулируется **разделом 13.9.1 Условий предоставления брокерских и иных услуг ПАО Сбербанк**, согласно которому банк предоставляет Инвесторам, которые совершили сделки по продаже ценных бумаг и/или валюты, возможность вывести денежные средства с Торгового брокерского счета без ожидания проведения расчётов по этим сделкам при согласии на использование сервиса «Вывод, не дожидаясь расчетов». 

**Источник данных**: раздел 13.9.1 контекста (brokerage_agreement).

Источники: brokerage_agreement
Время выполнения С кешем: 0.00101 сек.



In [62]:
# Считаем ускорение
if time_with_cache > 0:
    speedup = time_without_cache / time_with_cache
    print("=" * 50)
    print(f"ВЫВОД: Использование кеширования ускорило ответ системы в {speedup:.0f} раз!")

ВЫВОД: Использование кеширования ускорило ответ системы в 116312 раз!
